# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [15]:
%pip install -Uqqq langchain langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [16]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

# Simple Chain

In [ ]:
from langchain_core.prompts import PromptTemplate   # prompt 구성
from langchain.chat_models import init_chat_model   # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도')) # 프롬프트 템플릿 값이 1개일 경우만 사용
# 프롬프트 템플릿 변수가 2개 이상일 경우 dict형으로 전달
# print(chain.invoke(input={'city': '강원도', ...})) 

print()
print(chain.invoke(input={'city' : '강원도' }))

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 강원도 하면 가장 먼저 떠오르는 대표 농산물
- **옥수수**: 찰옥수수, 특히 홍천·정선 지역이 유명
- **메밀**: 봉평 메밀과 메밀국수·메밀전병
- **황태**: 인제 용대리 황태가 유명
- **오징어·명태 등 수산물**: 강릉·속초·동해 지역
- **곤드레**: 정선 곤드레나물과 곤드레밥
- **더덕·산나물·버섯**: 산간 지역에서 많이 생산
- **한우**: 횡성한우가 대표적
- **잣**: 홍천 잣
- **두릅과 아스파라거스**: 청정 산지 농산물

지역에 따라 특산물이 다르며, 특히 **횡성한우, 평창 메밀·한우, 정선 곤드레, 인제 황태, 홍천 옥수수·잣**이 널리 알려져 있습니다.

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 특히 평창·강릉·홍천 감자
- **옥수수**: 홍천·정선·강릉 찰옥수수
- **메밀**: 봉평 메밀과 메밀국수
- **황태**: 인제 용대리 황태
- **오징어**: 주문진·강릉 등 동해안 오징어
- **한우**: 횡성한우
- **더덕**: 횡성·홍천·정선 더덕
- **곤드레**: 정선 곤드레
- **송이버섯**: 양양 송이버섯
- **산나물**: 곰취, 취나물, 어수리 등

지역에 따라 특산물이 다르며, 특히 **감자·옥수수·메밀·황태·횡성한우**가 강원도를 대표하는 특산물로 많이 알려져 있습니다.


# Sequential Chain

In [18]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm 

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""

print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser

kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""

print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.\n\n이를 위해 먼저 문서 로더(document loader)를 사용하여 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서에 사용할 수 있는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 97, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 8, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIAGMYKoUdoHrcbBPzSEFYAO15qj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a04098-2559-74a3-a657-5c101fccc82a-0' tool_calls=[] invalid_tool_calls=[] usage_metad

In [19]:
# Sequential Chain
chain = chain1 | chain2

print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 등 외부 맥락에 직접 접근하기 어렵다는 한계가 있습니다. 이를 해결하려면 외부 데이터를 LLM에 연결해야 하며, LangChain은 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 자료를 불러올 수 있는 문서 로더를 제공합니다.


# Conditional Chain

In [20]:
# 조건에 따라 체인을 분기 실행해주는 Runnable
from langchain_core.runnables import RunnableBranch 

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 풀이과정 함께 작성해주세요. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수 (질문에 계산 또는 calc가 포함되면 수학 체인 선택)
def is_math_question(input_dict: dict) -> bool:
    # 입력받은 dict 에서 question 키의 값을 추출(없으면 빈 문자열)
    question: str = input_dict.get('question', '') 
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain), # True/False 결과 조건이 True 면 math_chain
    default_chain                   # False 면 default_chain
)

print(branch_chain.invoke({'question': '125*3 + 50 계산해줘.'}))

계산 과정:

1. \(125 \times 3 = 375\)
2. \(375 + 50 = 425\)

따라서 정답은 **425**입니다.


In [21]:
print(branch_chain.invoke({'question' : '나 오늘 우울해. 빵? 밥?'}))

오늘은 **따뜻한 밥** 먹자. 🍚  
속이 든든해지면 마음도 조금 덜 휑할 수 있어. 국이나 계란처럼 간단한 것도 좋아.  

근데 빵이 더 당긴다면 빵도 괜찮아—오늘은 **네가 먹고 싶은 걸 먹는 날**이야. 많이 힘들면 한입만이라도 먹어보자.


### Memory Chain

'RunnableWithMessageHistory' 를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [22]:
from langchain_core.chat_history import BaseChatMessageHistory # LANGCHAIN 대화기록 메모리 저장용 History 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage # 메시지 타입들
from pydantic import BaseModel, Field # Pydantic 모델(검증/기본값 생성) 도구
from typing import List # 타입 힌트(List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factory=list) : 인스턴스마다 독립적인 messages list를 구성
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_message(self, messages: List[BaseMessage]) -> None:
        # 전달 받은 메시지들을 기존 리스트 뒤에 추가
        self.messages.append(messages) 

    def clear(self) -> None:
        # 저장된 메시지들을 초기화
        self.messages = []

store = {} # {session_id : 히스토리 객체(InMemoryHistory)} 저장소

# 세션 ID 로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
        store[session_id] = InMemoryHistory() 
    return store[session_id] # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1') # 세션 ID '1'의 히스토리 기져오기 (없으면 메모리 공간 생성)
history1.add_messages([AIMessage(content='반갑습니다. Capybara님!')]) # AI 메시지 추가
history1.add_messages([HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!')]) # 유저메시지 추가

print(f"{history1 = }") # f"history1 = {history1}""

history2 = get_by_session_id('2') # 세션 ID '2'의 히스토리 기져오기
print(f"{history2 = }")


history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [23]:
# 채팅 프롬프트 템플릿 / 히스토리 자리표시자
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 실행시 히스토리를 붙여주는 Runnable 
from langchain_core.runnables import RunnableWithMessageHistory 

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'), # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain, 
    get_by_session_id, # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question', # 입력 dict 에서 question 키의 값은 사용자 메시지
    history_messages_key='history' # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': 'math', 
    'question':'민수는 강아지를 3마리 키우고 있습니다.'
}, config = { # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : '100' # 어떤 세션 히스토리 사용할지
    }
})

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='그렇군요. 민수는 강아지 3마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 38, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 27, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIAXwv9Hdyz8hCtcRGcZTvLIZVEK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04098-619f-7113-9332-25e77fe8fe77-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 57, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 27}})

In [24]:
chain_with_history.invoke({
    'domain': 'math', 
    'question':'소라는 고양이를 4마리 키우고 있습니다.'
}, config = { # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : '100' # 어떤 세션 히스토리 사용할지
    }
})

AIMessage(content='그렇군요. 소라는 고양이 4마리를 키우고 있네요. 민수와 소라가 키우는 동물은 모두 7마리입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 83, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 48, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIAYUfa97EXHFFP9Kq9R9cLyWZl0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04098-6969-7d40-9958-a66cc35d5392-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 96, 'total_tokens': 179, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 're

In [25]:
store # 현재 메모리에 저장된 세션별 대화 히스토리

{'1': InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})]),
 '2': InMemoryHistory(messages=[]),
 '100': InMemoryHistory(messages=[HumanMessage(content='민수는 강아지를 3마리 키우고 있습니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='그렇군요. 민수는 강아지 3마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 38, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 27, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIAX

### ChatMessageHistory

In [27]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory() # 기존 대화 내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id] # 해당 세션의 히스토리 객체 반환        

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'), # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain, 
    get_by_session_id, # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question', # 입력 dict 에서 question 키의 값은 사용자 메시지
    history_messages_key='history' # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': '심리상담', 
    'question':'요즘 갑자기 더워져서 짜증나는데? 나 성격 좋은데? 왜 이러지?'
}, config = { # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : '200' # 어떤 세션 히스토리 사용할지
    }
})

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='그럴 수 있어요. **더위 때문에 짜증이 난다고 해서 성격이 나빠진 건 전혀 아니에요.** 더위는 몸에 부담을 줘서 누구든 평소보다 예민해질 수 있습니다.\n\n- 체온을 낮추느라 몸이 계속 긴장함  \n- 땀·끈적임·답답함 같은 감각 자극이 늘어남  \n- 잠을 설쳐 피로와 인내심이 줄어듦  \n- 수분 부족이나 공복으로 두통·무기력·짜증이 생김  \n\n즉, “성격”보다는 **몸의 스트레스 반응**에 가까워요. 물을 조금씩 자주 마시고, 시원한 곳에서 쉬며, 얇고 통풍되는 옷을 입고, 가능하면 더운 시간대의 일정이나 갈등을 줄여보세요. 짜증이 올라올 때는 “내가 왜 이러지?”보다 “지금 몸이 너무 더운가?”라고 확인하는 것도 도움이 됩니다.\n\n다만 더위와 함께 **심한 어지럼, 구토, 의식이 흐려짐, 가슴 두근거림** 등이 있거나, 시원한 곳에서도 계속 과도하게 열감·짜증이 지속된다면 건강 상태를 확인해보는 게 좋아요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 372, 'prompt_tokens': 52, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 68, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerpr

In [28]:
chain_with_history.invoke({
    'domain': '심리상담', 
    'question':'그럼 너가 날씨를 좋게 만들어주면 되잖아'
}, config = { # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : '200' # 어떤 세션 히스토리 사용할지
    }
})

AIMessage(content='맞아요, 제가 날씨까지 바꿔드리면 참 좋겠네요 😅  \n아쉽게도 날씨를 직접 조절할 수는 없지만, **지금 덜 덥게 느끼도록 하는 방법**은 같이 찾아볼 수 있어요.\n\n지금은 우선 물을 마시고, 목·겨드랑이·손목처럼 혈관이 가까운 부위를 시원하게 해보세요. 에어컨이나 선풍기를 사용할 수 있다면 실내를 잠깐 식히고, 더운 시간대에는 무리한 활동을 줄이는 것도 좋아요.  \n\n그리고 오늘은 짜증이 난다고 해서 스스로에게 “성격이 왜 이래”라고 하지 마세요. **날씨가 나쁜 거지, 당신이 나쁜 사람이 된 건 아니니까요.**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 373, 'total_tokens': 555, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIEvSBtGL7R6GAYOKrIQQz2M5grm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0409c-8aca-7f30-a949-e2868a

In [31]:
print(store['200'])

Human: 요즘 갑자기 더워져서 짜증나는데? 나 성격 좋은데? 왜 이러지?
AI: 그럴 수 있어요. **더위 때문에 짜증이 난다고 해서 성격이 나빠진 건 전혀 아니에요.** 더위는 몸에 부담을 줘서 누구든 평소보다 예민해질 수 있습니다.

- 체온을 낮추느라 몸이 계속 긴장함  
- 땀·끈적임·답답함 같은 감각 자극이 늘어남  
- 잠을 설쳐 피로와 인내심이 줄어듦  
- 수분 부족이나 공복으로 두통·무기력·짜증이 생김  

즉, “성격”보다는 **몸의 스트레스 반응**에 가까워요. 물을 조금씩 자주 마시고, 시원한 곳에서 쉬며, 얇고 통풍되는 옷을 입고, 가능하면 더운 시간대의 일정이나 갈등을 줄여보세요. 짜증이 올라올 때는 “내가 왜 이러지?”보다 “지금 몸이 너무 더운가?”라고 확인하는 것도 도움이 됩니다.

다만 더위와 함께 **심한 어지럼, 구토, 의식이 흐려짐, 가슴 두근거림** 등이 있거나, 시원한 곳에서도 계속 과도하게 열감·짜증이 지속된다면 건강 상태를 확인해보는 게 좋아요.
Human: 그럼 너가 날씨를 좋게 만들어주면 되잖아
AI: 맞아요, 제가 날씨까지 바꿔드리면 참 좋겠네요 😅  
아쉽게도 날씨를 직접 조절할 수는 없지만, **지금 덜 덥게 느끼도록 하는 방법**은 같이 찾아볼 수 있어요.

지금은 우선 물을 마시고, 목·겨드랑이·손목처럼 혈관이 가까운 부위를 시원하게 해보세요. 에어컨이나 선풍기를 사용할 수 있다면 실내를 잠깐 식히고, 더운 시간대에는 무리한 활동을 줄이는 것도 좋아요.  

그리고 오늘은 짜증이 난다고 해서 스스로에게 “성격이 왜 이래”라고 하지 마세요. **날씨가 나쁜 거지, 당신이 나쁜 사람이 된 건 아니니까요.**


##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리